# True low-level leg control

This notebook uses `ll_sdk.LlSdk` to publish on `rt/lowcmd` while developer mode is active. The example moves the leg joints in a small gait-like cycle.

This bypasses the locomotion balance stack. Use only when the robot is safely supported, seated, or otherwise prevented from falling.


In [ ]:
import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
MODULES_DIR = NOTEBOOK_DIR.parent
ROOT_DIR = MODULES_DIR.parent
for path in (str(MODULES_DIR), str(ROOT_DIR), str(MODULES_DIR / "scripts")):
    if path not in sys.path:
        sys.path.insert(0, path)

IFACE = os.environ.get("G1_IFACE", "eth0")
DOMAIN_ID = int(os.environ.get("G1_DOMAIN_ID", "0"))
print(f"Configured for iface={IFACE!r}, domain_id={DOMAIN_ID}.")


Import the low-level SDK wrapper and UI helpers.


In [ ]:
import math
import threading
import time
import importlib

try:
    import hand_pose_navigation_copy  # noqa: F401
except ModuleNotFoundError:
    import hand_pose_navigation as _hand_pose_navigation
    sys.modules.setdefault("hand_pose_navigation_copy", _hand_pose_navigation)
    for _submodule in ("arm_ik", "arm_fk"):
        sys.modules.setdefault(
            f"hand_pose_navigation_copy.{_submodule}",
            importlib.import_module(f"hand_pose_navigation.{_submodule}"),
        )


import ipywidgets as widgets
from IPython.display import display

from ll_sdk import LLSdk


Create a gait controller. Every tick computes small sinusoidal offsets and ramps toward them before sending `move_ll_joint()`.


In [ ]:
LEFT_LEG = [0, 1, 2, 3, 4, 5]
RIGHT_LEG = [6, 7, 8, 9, 10, 11]

class LowLevelGaitDemo:
    def __init__(self, iface, domain_id):
        self.sdk = LLSdk(iface=iface, domain_id=domain_id)
        self.enabled = False
        self.dev_mode = False
        self.rate_hz = 35.0
        self.speed_rad_s = 0.45
        self.amplitude = 0.10
        self.frequency = 0.45
        self._neutral = None
        self._last = None
        self._stop = threading.Event()
        self._thread = None
        self._lock = threading.RLock()

    def enter_dev_mode(self):
        self.sdk.enter_dev_mode()
        self.dev_mode = True
        q = self.sdk.get_joint_positions()
        self._neutral = {i: q[i] for i in LEFT_LEG + RIGHT_LEG}
        self._last = dict(self._neutral)
        return "Developer mode entered and neutral leg pose captured."

    def start(self):
        if not self.dev_mode:
            raise RuntimeError("Press Enter Dev Mode before starting low-level gait.")
        self.enabled = True
        if self._thread is None or not self._thread.is_alive():
            self._stop.clear()
            self._thread = threading.Thread(target=self._loop, daemon=True)
            self._thread.start()
        return "Low-level gait loop running."

    def stop(self):
        self.enabled = False
        if self._neutral:
            self._ramp_to(self._neutral, duration=1.0)
        return "Gait loop stopped and ramped toward neutral."

    def _desired(self, t):
        assert self._neutral is not None
        a = float(self.amplitude)
        phase = 2.0 * math.pi * float(self.frequency) * t
        left = math.sin(phase)
        right = math.sin(phase + math.pi)
        target = dict(self._neutral)
        # hip pitch, knee, ankle pitch create a small stepping pattern
        for leg, s in ((LEFT_LEG, left), (RIGHT_LEG, right)):
            target[leg[2]] += 0.35 * a * s       # hip pitch
            target[leg[3]] += 1.00 * a * max(0.0, s)  # knee lift on swing phase
            target[leg[4]] -= 0.60 * a * max(0.0, s)  # ankle pitch compensation
            target[leg[1]] += 0.20 * a * s       # small roll shift
        return target

    def _ramped(self, desired, dt):
        assert self._last is not None
        max_step = max(0.001, self.speed_rad_s * dt)
        out = {}
        for joint, des in desired.items():
            cur = self._last[joint]
            step = max(-max_step, min(max_step, des - cur))
            out[joint] = cur + step
        self._last = dict(out)
        return out

    def _ramp_to(self, desired, duration=1.0):
        start = time.monotonic()
        dt = 1.0 / max(1.0, self.rate_hz)
        while time.monotonic() - start < duration:
            targets = self._ramped(desired, dt)
            self.sdk.move_ll_joint(targets)
            time.sleep(dt)

    def _loop(self):
        start = time.monotonic()
        dt = 1.0 / max(1.0, self.rate_hz)
        while not self._stop.is_set():
            if self.enabled:
                with self._lock:
                    desired = self._desired(time.monotonic() - start)
                    targets = self._ramped(desired, dt)
                    self.sdk.move_ll_joint(targets)
            time.sleep(dt)

gait = LowLevelGaitDemo(IFACE, DOMAIN_ID)
print("Low-level gait demo ready. Do not start until the robot is supported.")


Run the controls. Enter developer mode first, then start the gait loop. Keep amplitudes small.


In [ ]:
amplitude = widgets.FloatSlider(value=0.10, min=0.01, max=0.25, step=0.01, description="Amplitude")
frequency = widgets.FloatSlider(value=0.45, min=0.05, max=1.0, step=0.05, description="Hz")
speed = widgets.FloatSlider(value=0.45, min=0.05, max=1.5, step=0.05, description="Ramp")
enter_dev = widgets.Button(description="Enter Dev Mode", button_style="warning")
start = widgets.Button(description="Start Gait", button_style="success")
stop = widgets.Button(description="Stop", button_style="danger")
status = widgets.HTML(value="")


def sync_params():
    gait.amplitude = float(amplitude.value)
    gait.frequency = float(frequency.value)
    gait.speed_rad_s = float(speed.value)


def on_enter(_):
    try:
        status.value = gait.enter_dev_mode()
    except Exception as exc:
        status.value = f"Enter dev mode failed: {exc}"


def on_start(_):
    try:
        sync_params()
        status.value = gait.start()
    except Exception as exc:
        status.value = f"Start failed: {exc}"


def on_stop(_):
    try:
        status.value = gait.stop()
    except Exception as exc:
        status.value = f"Stop failed: {exc}"

for w in (amplitude, frequency, speed):
    w.observe(lambda _change: sync_params(), names="value")
enter_dev.on_click(on_enter)
start.on_click(on_start)
stop.on_click(on_stop)
display(widgets.VBox([widgets.HBox([amplitude, frequency, speed]), widgets.HBox([enter_dev, start, stop]), status]))
